# Polymarket Analyst

## Description
Polymarket Analyst retrieves live Polymarket market/event data, normalizes odds and liquidity fields, runs probability/edge analysis, performs lightweight historical/backtest-style strategy checks when data is available, identifies disputed or proposed market resolutions with arbitrage potential, and returns evidence-backed market research. Use when a task requires finding relevant prediction markets, comparing prices to an outside probability estimate, ranking markets by liquidity/volume/edge, analyzing event contracts, sanity-checking a Polymarket investment thesis, or prototyping/backtesting event-driven trading strategies (including Kelly allocation suggestions) without executing trades.

## System Prompt
You are the Polymarket Analyst agent. Your responsibility is to perform evidence-backed Polymarket research and strategy analysis in a temporary notebook copy, then return a concise summary to the parent agent.

Inputs you may receive from the parent agent:
- A market/event search query, topic, slug, tag, event type, or time horizon.
- A probability estimate or outside model output to compare against Polymarket prices.
- A request to rank markets by liquidity, volume, price, implied probability, or estimated edge.
- A request to find markets with active proposed resolutions or potential settlement premiums (disputed markets).
- Historical market snapshots or trades supplied by the parent for backtesting.
- Constraints such as minimum liquidity, maximum position size, acceptable risk, or categories to exclude.

Default workflow:
1. Clarify the analysis objective from the task and identify required data: active/closed events, markets, outcomes, prices, volume, liquidity, timestamps, or provided historical datasets.
2. Use live public APIs where possible, especially Polymarket Gamma (`https://gamma-api.polymarket.com`) and, if needed, CLOB endpoints. Do not use private keys or authenticated trading endpoints.
3. Normalize API responses into pandas DataFrames using the helper cells in this notebook before analyzing.
4. Identify disputed, proposed, settled, etc. markets based on their UMA resolution.
5. Validate data quality: check missing fields, malformed JSON-encoded outcome/price lists, market status, liquidity, volume, and timestamp freshness.
6. Run the requested analysis: market search, odds/implied probability interpretation, edge calculation against an external probability, ranking/filtering, sensitivity analysis, or backtest-style simulation on provided historical data.
7. Prefer conservative conclusions. Distinguish observed market prices from model estimates and subjective assumptions.
8. For any identified edge, calculate recommended allocation weights using the Kelly criteria (typically fractional Kelly, e.g., 0.25x or 0.5x, to account for uncertainty).
9. Return a concise parent-facing summary with citations to live endpoints or datasets used, important numbers, caveats, and recommended next steps.
10. Whenever mentioning any specific Polymarket market or event in the final response, include its direct Polymarket URL in the same bullet/sentence. Prefer a URL from API fields if available; otherwise construct it from `slug`, `market_slug`, `eventSlug`, or equivalent fields as `https://polymarket.com/event/<event-slug>` or `https://polymarket.com/market/<market-slug>`. If no reliable slug/URL is available, explicitly write `URL unavailable` rather than omitting it.

Safety and constraints:
- Never execute real trades, sign transactions, request private keys, or provide instructions that require custody of user funds.
- Do not invent data. If an API request fails or data is unavailable, say so and provide a reproducible fallback or next step.
- Treat investment recommendations as informational analysis, not financial advice.
- Avoid overfitting in backtests; state assumptions, fees/slippage if modeled, sample size, and limitations.
- Keep reusable source notebooks free of secrets and one-off user data.

Final response format to the parent agent:
- **Objective**: one sentence describing the task.
- **Data used**: endpoints/files, filters, row counts, and freshness if available.
- **Key findings**: bullets with market/event name, direct Polymarket URL, prices/probabilities/liquidity/volume/edge/backtest metrics.
- **Recommendation or interpretation**: actionable but caveated conclusion; include the direct Polymarket URL next to each market mentioned.
- **Kelly Allocation**: suggested fraction of bankroll based on Kelly criteria for any identified edge; include the direct Polymarket URL for each market allocation.
- **Caveats / next steps**: missing data, assumptions, and what to verify before acting.

In [14]:
import ast
import json
import math
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import requests

# Public Polymarket Gamma API endpoint. These helpers never use authenticated trading endpoints.
GAMMA_API_URL = "https://gamma-api.polymarket.com"
DEFAULT_TIMEOUT = 20

session = requests.Session()
session.headers.update({"User-Agent": "orion-polymarket-analyst/1.0"})


def gamma_get(path: str, params: Optional[Dict[str, Any]] = None, timeout: int = DEFAULT_TIMEOUT) -> Any:
    """GET a public Gamma API path and return parsed JSON with useful error context."""
    url = f"{GAMMA_API_URL}{path if path.startswith('/') else '/' + path}"
    response = session.get(url, params=params or {}, timeout=timeout)
    response.raise_for_status()
    return response.json()


def fetch_events(
    *,
    active: Optional[bool] = True,
    closed: Optional[bool] = False,
    limit: int = 50,
    offset: int = 0,
    search: Optional[str] = None,
    slug: Optional[str] = None,
    order: str = "volume24hr",
    ascending: bool = False,
) -> List[Dict[str, Any]]:
    """Fetch events from Gamma. Supports broad active-event discovery and text/slug search."""
    params: Dict[str, Any] = {"limit": limit, "offset": offset, "order": order, "ascending": str(ascending).lower()}
    if active is not None:
        params["active"] = str(active).lower()
    if closed is not None:
        params["closed"] = str(closed).lower()
    if search:
        params["search"] = search
    if slug:
        params["slug"] = slug
    return gamma_get("/events", params=params)


def fetch_active_events(limit: int = 50, offset: int = 0) -> List[Dict[str, Any]]:
    """Backward-compatible helper: fetch active, non-closed events."""
    return fetch_events(active=True, closed=False, limit=limit, offset=offset)


def _parse_maybe_json_list(value: Any) -> List[Any]:
    """Parse Gamma fields that may arrive as JSON strings, Python-list strings, or native lists."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return []
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(stripped)
                return parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                continue
        return [stripped]
    return [value]


def _to_float(value: Any, default: float = np.nan) -> float:
    """Safely convert mixed API numeric fields to float."""
    try:
        if value is None or value == "":
            return default
        return float(value)
    except Exception:
        return default


def normalize_events(events: Sequence[Dict[str, Any]]) -> pd.DataFrame:
    """Flatten event-level fields into one row per event."""
    rows = []
    for event in events:
        markets = event.get("markets") or []
        rows.append(
            {
                "event_id": event.get("id"),
                "event_slug": event.get("slug"),
                "event_title": event.get("title"),
                "category": event.get("category"),
                "active": event.get("active"),
                "closed": event.get("closed"),
                "start_date": event.get("startDate"),
                "end_date": event.get("endDate"),
                "volume": _to_float(event.get("volume")),
                "volume24hr": _to_float(event.get("volume24hr")),
                "liquidity": _to_float(event.get("liquidity")),
                "market_count": len(markets),
            }
        )
    return pd.DataFrame(rows)


def normalize_markets(events: Sequence[Dict[str, Any]]) -> pd.DataFrame:
    """Flatten Gamma events into one row per market with parsed outcomes and prices."""
    rows = []
    for event in events:
        for market in event.get("markets") or []:
            outcomes = _parse_maybe_json_list(market.get("outcomes"))
            prices = [_to_float(x) for x in _parse_maybe_json_list(market.get("outcomePrices"))]
            outcome_price_map = dict(zip(outcomes, prices)) if outcomes and prices else {}
            rows.append(
                {
                    "event_id": event.get("id"),
                    "event_slug": event.get("slug"),
                    "event_title": event.get("title"),
                    "market_id": market.get("id"),
                    "condition_id": market.get("conditionId"),
                    "question": market.get("question"),
                    "market_slug": market.get("slug"),
                    "active": market.get("active"),
                    "closed": market.get("closed"),
                    "archived": market.get("archived"),
                    "end_date": market.get("endDate") or event.get("endDate"),
                    "volume": _to_float(market.get("volume")),
                    "volume24hr": _to_float(market.get("volume24hr")),
                    "liquidity": _to_float(market.get("liquidity")),
                    "best_bid": _to_float(market.get("bestBid")),
                    "best_ask": _to_float(market.get("bestAsk")),
                    "last_trade_price": _to_float(market.get("lastTradePrice")),
                    "outcomes": outcomes,
                    "outcome_prices": prices,
                    "outcome_price_map": outcome_price_map,
                    "yes_price": outcome_price_map.get("Yes", prices[0] if prices else np.nan),
                    "no_price": outcome_price_map.get("No", prices[1] if len(prices) > 1 else np.nan),
                    "uma_resolution_statuses": _parse_maybe_json_list(market.get("umaResolutionStatuses")),
                }
            )
    return pd.DataFrame(rows)


def summarize_market_universe(markets_df: pd.DataFrame) -> pd.DataFrame:
    """Return a compact market universe summary sorted by recent activity/liquidity."""
    if markets_df.empty:
        return markets_df
    cols = [
        "event_title",
        "question",
        "yes_price",
        "no_price",
        "volume24hr",
        "volume",
        "liquidity",
        "end_date",
        "market_slug",
    ]
    available_cols = [c for c in cols if c in markets_df.columns]
    return (
        markets_df.loc[:, available_cols]
        .sort_values(["volume24hr", "liquidity", "volume"], ascending=False, na_position="last")
        .reset_index(drop=True)
    )


def calculate_binary_edge(
    markets_df: pd.DataFrame,
    probability_estimates: Dict[str, float],
    *,
    price_col: str = "yes_price",
    market_key: str = "market_slug",
) -> pd.DataFrame:
    """Compare external fair probabilities to current YES prices and compute expected value per $1 payout share.

    probability_estimates maps market_slug (or the chosen market_key) to fair YES probability in [0, 1].
    For a binary YES share costing p and paying $1 if correct, expected profit per share is fair_prob - p.
    ROI on cost is (fair_prob - p) / p, before fees/slippage/spread.
    """
    if markets_df.empty:
        return markets_df.copy()
    analysis = markets_df.copy()
    analysis["fair_prob"] = analysis[market_key].map(probability_estimates)
    analysis["market_price"] = pd.to_numeric(analysis[price_col], errors="coerce")
    analysis["edge_per_share"] = analysis["fair_prob"] - analysis["market_price"]
    analysis["roi_on_cost"] = analysis["edge_per_share"] / analysis["market_price"]
    analysis["kelly_fraction_binary"] = (
        (analysis["fair_prob"] - analysis["market_price"]) / (1 - analysis["market_price"])
    ).clip(lower=0)
    return analysis.sort_values("edge_per_share", ascending=False, na_position="last")


def backtest_threshold_strategy(
    snapshots: pd.DataFrame,
    *,
    fair_prob_col: str = "fair_prob",
    price_col: str = "market_price",
    resolved_col: str = "resolved_yes",
    min_edge: float = 0.05,
    stake: float = 1.0,
) -> Dict[str, Any]:
    """Toy backtest: buy YES when fair_prob - price >= min_edge; payout is resolved_yes * stake / price.

    Expects one row per historical opportunity with market_price in (0, 1), fair probability, and binary resolution.
    Returns simple gross metrics before fees/slippage and without order-book fill constraints.
    """
    required = {fair_prob_col, price_col, resolved_col}
    missing = required - set(snapshots.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    trades = snapshots.copy()
    trades["edge"] = trades[fair_prob_col] - trades[price_col]
    trades = trades[(trades["edge"] >= min_edge) & (trades[price_col] > 0) & (trades[price_col] < 1)].copy()
    if trades.empty:
        return {"n_trades": 0, "total_staked": 0.0, "pnl": 0.0, "roi": np.nan, "win_rate": np.nan, "trades": trades}

    trades["stake"] = stake
    trades["shares"] = trades["stake"] / trades[price_col]
    trades["payout"] = trades["shares"] * trades[resolved_col].astype(float)
    trades["pnl"] = trades["payout"] - trades["stake"]
    total_staked = trades["stake"].sum()
    pnl = trades["pnl"].sum()
    return {
        "n_trades": int(len(trades)),
        "total_staked": float(total_staked),
        "pnl": float(pnl),
        "roi": float(pnl / total_staked) if total_staked else np.nan,
        "win_rate": float(trades[resolved_col].mean()),
        "avg_edge": float(trades["edge"].mean()),
        "trades": trades,
    }


def fetch_markets_direct(
    *,
    active: bool = True,
    closed: bool = False,
    limit: int = 100,
    offset: int = 0,
    order: str = "volume24hr",
    ascending: bool = False,
    min_volume24h: Optional[float] = None,
) -> List[Dict[str, Any]]:
    """Fetch markets directly from the /markets endpoint, optionally paginating until min_volume24h is reached."""
    all_markets = []
    current_offset = offset
    
    while True:
        params = {
            "limit": limit,
            "offset": current_offset,
            "active": str(active).lower(),
            "closed": str(closed).lower(),
            "order": order,
            "ascending": str(ascending).lower(),
        }
        batch = gamma_get("/markets", params=params)
        if not batch:
            break
            
        for market in batch:
            vol = _to_float(market.get("volume24hr"))
            if min_volume24h is not None and (pd.isna(vol) or vol < min_volume24h):
                return all_markets
            all_markets.append(market)
            
        if len(batch) < limit:
            break
        current_offset += limit
        
    return all_markets


def build_proposed_markets_table(markets: List[Dict[str, Any]]) -> pd.DataFrame:
    """Filter for markets with an active proposed resolution and calculate arbitrage premiums."""
    records = []
    for market in markets:
        statuses = _parse_maybe_json_list(market.get("umaResolutionStatuses"))
        statuses_lower = [str(s).lower() for s in statuses]
        
        # Look for 'proposed' but not 'resolved' or 'settled'
        if "proposed" not in statuses_lower or any(s in statuses_lower for s in ["resolved", "settled"]):
            continue
            
        last_price = _to_float(market.get("lastTradePrice"))
        best_bid = _to_float(market.get("bestBid"))
        best_ask = _to_float(market.get("bestAsk"))
        
        if pd.isna(last_price) or pd.isna(best_bid) or pd.isna(best_ask):
            continue
            
        # Price-based heuristic for implied proposal
        implied_yes = last_price >= 0.50
        premium_cents = (1.0 - last_price) * 100 if implied_yes else last_price * 100
        
        slug = market.get("slug")
        records.append({
            "market_id": market.get("id"),
            "question": market.get("question"),
            "slug": slug,
            "url": f"https://polymarket.com/market/{slug}" if slug else None,
            "uma_status": ", ".join(statuses),
            "implied_proposal": "Yes" if implied_yes else "No",
            "last_price": last_price,
            "premium_cents": premium_cents,
            "best_bid": best_bid,
            "best_ask": best_ask,
            "volume_24h": _to_float(market.get("volume24hr")),
            "liquidity": _to_float(market.get("liquidity")),
        })
        
    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values(["premium_cents", "volume_24h"], ascending=[False, False]).reset_index(drop=True)
    return df


print("Polymarket Analyst helpers initialized, including Disputed Market Analysis.")

Polymarket Analyst helpers initialized, including Disputed Market Analysis.


In [8]:
# Refinements: safer edge math and local keyword filtering for broad API search results.

def filter_markets_by_text(markets_df: pd.DataFrame, query: str, columns: Sequence[str] = ("event_title", "question", "market_slug")) -> pd.DataFrame:
    """Filter normalized markets to rows where query appears in selected text columns."""
    if markets_df.empty or not query:
        return markets_df.copy()
    available_cols = [col for col in columns if col in markets_df.columns]
    if not available_cols:
        return markets_df.copy()
    text = markets_df[available_cols].fillna("").astype(str).agg(" ".join, axis=1)
    return markets_df[text.str.contains(query, case=False, regex=False, na=False)].copy()


def calculate_binary_edge(
    markets_df: pd.DataFrame,
    probability_estimates: Dict[str, float],
    *,
    price_col: str = "yes_price",
    market_key: str = "market_slug",
    min_price: float = 0.001,
    max_price: float = 0.999,
) -> pd.DataFrame:
    """Compare external fair probabilities to YES prices with robust handling of 0/1/stale prices.

    Rows outside (min_price, max_price) keep edge_per_share but receive NaN ROI/Kelly because sizing math is unstable
    at exact 0/1 prices and often indicates stale, suspended, or unfillable markets.
    """
    if markets_df.empty:
        return markets_df.copy()

    analysis = markets_df.copy()
    analysis["fair_prob"] = pd.to_numeric(analysis[market_key].map(probability_estimates), errors="coerce").clip(0, 1)
    analysis["market_price"] = pd.to_numeric(analysis[price_col], errors="coerce")
    valid_price = analysis["market_price"].between(min_price, max_price, inclusive="both")
    analysis["edge_per_share"] = analysis["fair_prob"] - analysis["market_price"]
    analysis["roi_on_cost"] = np.where(valid_price, analysis["edge_per_share"] / analysis["market_price"], np.nan)
    analysis["kelly_fraction_binary"] = np.where(
        valid_price,
        ((analysis["fair_prob"] - analysis["market_price"]) / (1 - analysis["market_price"])).clip(lower=0),
        np.nan,
    )
    analysis["price_quality_flag"] = np.where(valid_price, "ok", "invalid_or_stale_for_sizing")
    return analysis.sort_values("edge_per_share", ascending=False, na_position="last")

print("Refinement helpers loaded: filter_markets_by_text() and safer calculate_binary_edge().")

Refinement helpers loaded: filter_markets_by_text() and safer calculate_binary_edge().


## Use cases and reusable workflows

Use this sub-agent for concrete Polymarket analysis tasks such as:

1. **Market discovery** — “Find active AI, election, Fed, crypto, or sports markets with meaningful volume/liquidity.”
2. **Odds interpretation** — “Summarize current implied probabilities and liquidity for this event or slug.”
3. **Edge analysis** — “Compare my model probability to Polymarket’s YES price and rank possible mispricings.”
4. **Position sizing** — “Suggest allocation weights based on the Kelly criteria for identified edges, accounting for risk and uncertainty.”
5. **Disputed Market Analysis** — “Find markets with active proposed resolutions and calculate the potential arbitrage premium.”
6. **Portfolio screening** — “Filter markets by minimum liquidity, ending date, volume, price range, and topic.”
7. **Strategy simulation** — “Given historical snapshots and resolutions, test a threshold strategy before considering any real-world action.”
8. **Risk review** — “Explain caveats: binary payout profile, resolution risk, spread/slippage, stale prices, correlated markets, and sample-size limits.”

The examples below are safe: they use public read-only endpoints or synthetic data and never place trades.

In [9]:
# Example 1: discover active markets by keyword/topic.
# Change QUERY to a topic such as "AI", "election", "Fed", "Bitcoin", or a specific event name.
QUERY = "AI"

try:
    events = fetch_events(search=QUERY, active=True, closed=False, limit=25)
    events_df = normalize_events(events)
    markets_df_raw = normalize_markets(events)
    markets_df = filter_markets_by_text(markets_df_raw, QUERY)
    if markets_df.empty:
        markets_df = markets_df_raw
        print("No local text matches found; showing broad API results instead.")
    print(
        f"Fetched {len(events)} events and normalized {len(markets_df_raw)} markets for query={QUERY!r}; "
        f"showing {len(markets_df)} locally matched/broad markets."
    )
    display(summarize_market_universe(markets_df).head(10))
except Exception as exc:
    print(f"Live API example failed: {type(exc).__name__}: {exc}")
    print("This can happen due to network/API availability. Helper functions remain usable with provided data.")

Fetched 25 events and normalized 782 markets for query='AI'; showing 32 locally matched/broad markets.


,event_title,question,yes_price,no_price,volume24hr,volume,liquidity,end_date,market_slug
0,Strait of Hormuz traffic returns to normal by ...,Strait of Hormuz traffic returns to normal by ...,0.0550,0.9450,1.546859e+06,8.792955e+06,1.844993e+05,2026-05-15T00:00:00Z,strait-of-hormuz-traffic-returns-to-normal-by-...
1,Strait of Hormuz traffic returns to normal by ...,Strait of Hormuz traffic returns to normal by ...,0.2750,0.7250,1.324919e+06,8.164024e+06,3.111305e+05,2026-05-31T00:00:00Z,strait-of-hormuz-traffic-returns-to-normal-by-...
2,Iran closes its airspace by...?,Iran closes its airspace by May 8?,0.0315,0.9685,1.286006e+06,5.598310e+06,2.265104e+05,2026-05-31T00:00:00Z,iran-closes-its-airspace-by-may-8-754-861
3,FC Bayern München vs. Paris Saint-Germain FC,Will FC Bayern München win on 2026-05-06?,0.6050,0.3950,8.550210e+05,1.347628e+06,2.443874e+06,2026-05-06T19:00:00Z,ucl-bay-psg-2026-05-06-bay
4,FC Bayern München vs. Paris Saint-Germain FC,Will Paris Saint-Germain FC win on 2026-05-06?,0.2250,0.7750,7.394947e+05,1.187515e+06,2.439278e+06,2026-05-06T19:00:00Z,ucl-bay-psg-2026-05-06-psg
5,Iran closes its airspace by...?,Iran closes its airspace by May 31?,0.2850,0.7150,6.531132e+05,1.251086e+06,1.273086e+05,2026-05-31T00:00:00Z,iran-closes-its-airspace-by-may-31-434-443-672
6,Trump announces US blockade of Hormuz lifted b...,Will Donald Trump announce that the United Sta...,0.0485,0.9515,4.230218e+05,1.211347e+06,6.570669e+04,2026-05-08T00:00:00Z,will-donald-trump-announce-that-the-united-sta...
7,Trump announces US blockade of Hormuz lifted b...,Will Donald Trump announce that the United Sta...,0.2850,0.7150,3.961489e+05,1.412729e+06,3.871560e+04,2026-05-15T00:00:00Z,will-donald-trump-announce-that-the-united-sta...
8,Iran closes its airspace by...?,Iran closes its airspace by May 15?,0.1350,0.8650,3.376567e+05,3.376587e+05,6.792953e+04,2026-05-15T00:00:00Z,iran-closes-its-airspace-by-may-15-844-228
9,Trump announces US blockade of Hormuz lifted b...,Will Donald Trump announce that the United Sta...,0.4800,0.5200,2.118323e+05,2.190907e+06,8.144434e+04,2026-05-31T00:00:00Z,will-donald-trump-announce-that-the-united-sta...


In [10]:
# Example 2: compare external fair probabilities to current YES prices.
# In a real task, probability_estimates should come from a model, forecast, or user-provided assumptions.

if "markets_df" in globals() and not markets_df.empty and "market_slug" in markets_df.columns:
    liquid_markets = markets_df.dropna(subset=["market_slug", "yes_price"]).copy()
    liquid_markets = liquid_markets[liquid_markets["yes_price"].between(0.001, 0.999, inclusive="both")]
    candidate_markets = liquid_markets.head(3).copy()
    probability_estimates = {
        row.market_slug: min(max(float(row.yes_price) + 0.03, 0.01), 0.99)
        for row in candidate_markets.itertuples()
    }
    edge_df = calculate_binary_edge(candidate_markets, probability_estimates)
    display(edge_df[["question", "market_slug", "yes_price", "fair_prob", "edge_per_share", "roi_on_cost", "kelly_fraction_binary", "price_quality_flag"]])
else:
    print("Run Example 1 first or provide a markets_df DataFrame.")

,question,market_slug,yes_price,fair_prob,edge_per_share,roi_on_cost,kelly_fraction_binary,price_quality_flag
5,Will Spain win the 2026 FIFA World Cup?,will-spain-win-the-2026-fifa-world-cup-963,0.1535,0.1835,0.03,0.195440,0.035440,ok
12,Will Haiti win the 2026 FIFA World Cup?,will-haiti-win-the-2026-fifa-world-cup,0.0015,0.0315,0.03,20.000000,0.030045,ok
92,Will Gina Raimondo win the 2028 Democratic pre...,will-gina-raimondo-win-the-2028-democratic-pre...,0.0065,0.0365,0.03,4.615385,0.030196,ok


In [11]:
# Example 3: toy threshold-strategy backtest on synthetic historical snapshots.
# This demonstrates the expected input shape for user-provided historical data.

synthetic_snapshots = pd.DataFrame(
    {
        "market_slug": ["m1", "m2", "m3", "m4", "m5"],
        "timestamp": pd.date_range("2024-01-01", periods=5, freq="D"),
        "market_price": [0.40, 0.55, 0.30, 0.70, 0.20],
        "fair_prob": [0.50, 0.57, 0.42, 0.68, 0.33],
        "resolved_yes": [1, 0, 1, 1, 0],
    }
)

backtest_result = backtest_threshold_strategy(synthetic_snapshots, min_edge=0.08, stake=10.0)
print({k: v for k, v in backtest_result.items() if k != "trades"})
display(backtest_result["trades"])

{'n_trades': 3, 'total_staked': 30.0, 'pnl': 28.333333333333336, 'roi': 0.9444444444444445, 'win_rate': 0.6666666666666666, 'avg_edge': 0.11666666666666665}


,market_slug,timestamp,market_price,fair_prob,resolved_yes,edge,stake,shares,payout,pnl
0,m1,2024-01-01,0.4,0.50,1,0.10,10.0,25.000000,25.000000,15.000000
2,m3,2024-01-03,0.3,0.42,1,0.12,10.0,33.333333,33.333333,23.333333
4,m5,2024-01-05,0.2,0.33,0,0.13,10.0,50.000000,0.000000,-10.000000


In [ ]:
# Example 4: Disputed/Proposed Market Analysis
# This finds markets that have a proposed resolution but are not yet settled, 
# calculating the potential arbitrage premium.

try:
    # Fetch liquid markets
    liquid_markets = fetch_markets_direct(min_volume24h=250.0)
    proposed_df = build_proposed_markets_table(liquid_markets)

    print(f"Total liquid markets fetched: {len(liquid_markets)}")
    print(f"Active proposed markets found: {len(proposed_df)}")

    if not proposed_df.empty:
        display(proposed_df[['question', 'implied_proposal', 'last_price', 'premium_cents', 'volume_24h', 'url']].head(10))
    else:
        print("No active proposed markets found at this time.")
except Exception as exc:
    print(f"Disputed market analysis failed: {type(exc).__name__}: {exc}")

## Validation tests

These tests verify that reusable helpers work on representative API-like payloads and that the live public API is reachable. If the live API test fails due to connectivity, inspect the error and continue with supplied datasets or retry later.

In [12]:
def run_unit_tests() -> None:
    """Fast local tests for parsing, normalization, filtering, edge calculations, and toy backtest logic."""
    sample_events = [
        {
            "id": "event-1",
            "slug": "sample-event",
            "title": "Sample AI Event",
            "active": True,
            "closed": False,
            "volume": "1000.5",
            "volume24hr": "125.25",
            "liquidity": "500",
            "markets": [
                {
                    "id": "market-1",
                    "conditionId": "cond-1",
                    "question": "Will the sample AI market resolve yes?",
                    "slug": "sample-market",
                    "active": True,
                    "closed": False,
                    "outcomes": '["Yes", "No"]',
                    "outcomePrices": '["0.42", "0.58"]',
                    "volume": "900",
                    "volume24hr": "100",
                    "liquidity": "450",
                }
            ],
        }
    ]

    events_df = normalize_events(sample_events)
    markets_df = normalize_markets(sample_events)
    assert len(events_df) == 1
    assert len(markets_df) == 1
    assert len(filter_markets_by_text(markets_df, "AI")) == 1
    assert markets_df.loc[0, "outcomes"] == ["Yes", "No"]
    assert math.isclose(markets_df.loc[0, "yes_price"], 0.42)
    assert math.isclose(markets_df.loc[0, "no_price"], 0.58)

    edge_df = calculate_binary_edge(markets_df, {"sample-market": 0.50})
    assert math.isclose(edge_df.iloc[0]["edge_per_share"], 0.08)
    assert edge_df.iloc[0]["kelly_fraction_binary"] > 0
    assert edge_df.iloc[0]["price_quality_flag"] == "ok"

    zero_price_df = markets_df.copy()
    zero_price_df["yes_price"] = 0.0
    zero_edge_df = calculate_binary_edge(zero_price_df, {"sample-market": 0.10})
    assert np.isnan(zero_edge_df.iloc[0]["roi_on_cost"])
    assert zero_edge_df.iloc[0]["price_quality_flag"] == "invalid_or_stale_for_sizing"

    snapshots = pd.DataFrame(
        {
            "market_price": [0.40, 0.60, 0.20],
            "fair_prob": [0.50, 0.61, 0.35],
            "resolved_yes": [1, 0, 0],
        }
    )
    result = backtest_threshold_strategy(snapshots, min_edge=0.05, stake=1.0)
    assert result["n_trades"] == 2
    assert "trades" in result and isinstance(result["trades"], pd.DataFrame)

    print("All local unit tests passed.")


run_unit_tests()

All local unit tests passed.


In [13]:
def run_live_api_smoke_test(limit: int = 3) -> None:
    """Small read-only live API smoke test. This should not fail the notebook if the network is unavailable."""
    try:
        events = fetch_active_events(limit=limit)
        assert isinstance(events, list), "Expected Gamma /events response to be a list"
        events_df = normalize_events(events)
        markets_df = normalize_markets(events)
        print(
            f"Live API smoke test passed: {len(events)} events, "
            f"{len(markets_df)} markets, fetched at {datetime.now(timezone.utc).isoformat()}"
        )
        if not markets_df.empty:
            display(summarize_market_universe(markets_df).head(limit))
    except Exception as exc:
        print(f"Live API smoke test skipped/failed: {type(exc).__name__}: {exc}")
        print("Local unit tests still validate the reusable helpers; retry live API later if needed.")


run_live_api_smoke_test()

Live API smoke test passed: 3 events, 72 markets, fetched at 2026-05-06T16:09:48.796192+00:00


,event_title,question,yes_price,no_price,volume24hr,volume,liquidity,end_date,market_slug
0,When will Bitcoin hit $150k?,"Will Bitcoin hit $150k by June 30, 2026?",0.0135,0.9865,5.821653e+06,1.573401e+07,19822.55555,2026-07-01T04:00:00Z,will-bitcoin-hit-150k-by-june-30-2026
1,US x Iran permanent peace deal by...?,"US x Iran permanent peace deal by May 15, 2026?",0.1470,0.8530,2.193323e+06,5.510302e+06,268976.61588,2026-05-15T00:00:00Z,us-x-iran-permanent-peace-deal-by-may-15-2026
2,US x Iran permanent peace deal by...?,"US x Iran permanent peace deal by May 31, 2026?",0.2450,0.7550,1.989970e+06,1.356424e+07,294634.78660,2026-05-31T00:00:00Z,us-x-iran-permanent-peace-deal-by-may-31-2026-...
